# Python Database SQLite3 Connectivity

## What you will learn
- What a database is and why SQLite is useful
- How to connect to SQLite using Python's built-in `sqlite3` module
- How to create databases, tables, and constraints
- How to insert, read, update, and delete rows
- How to use `fetchone()`, `fetchmany()`, and `fetchall()`
- How to use transactions, commits, rollbacks, and error handling
- How to design schemas, joins, aggregations, and reusable helper functions
- How to avoid common mistakes and write safe SQL
- How to build mini practical examples with SQLite
- How SQLite concepts prepare you for larger databases later

# Introduction

## What is it?
`sqlite3` is Python's built-in module for working with SQLite databases. SQLite is a lightweight relational database engine that stores data in a local file or in memory.

## Why is it used?
- It is included with Python, so beginners can start immediately
- It is excellent for learning SQL and database fundamentals
- It is useful for local apps, prototypes, tools, and testing
- It allows structured, searchable, and persistent data storage

## Real-world relevance
- Small desktop applications
- Personal productivity tools
- Local analytics projects
- Testing environments
- Educational projects and prototypes
- Local caches for larger systems

# Basic Syntax Explanation

## Usual workflow
1. Import `sqlite3`
2. Create a connection using `sqlite3.connect(...)`
3. Create a cursor using `connection.cursor()`
4. Run SQL commands with `cursor.execute(...)`
5. Save changes using `connection.commit()` when data changes
6. Read results using fetch methods
7. Close the cursor and connection

## Important operators and syntax
- Python assignment: `=`
- SQL comparison: `=`, `>`, `<`, `!=`
- SQL logic: `AND`, `OR`, `IN`, `LIKE`
- SQLite placeholder syntax: `?`

## Safety rule
Always use parameterized queries with placeholders instead of inserting raw user input directly into SQL strings.

In [ ]:
# Import the built-in sqlite3 module.
import sqlite3

# Create an in-memory database for a quick and safe example.
connection = sqlite3.connect(':memory:')

# Create a cursor to run SQL commands.
cursor = connection.cursor()

# Create a simple learners table.
cursor.execute('CREATE TABLE learners (id INTEGER PRIMARY KEY, name TEXT, score INTEGER)')

# Insert one row using placeholders.
cursor.execute('INSERT INTO learners (name, score) VALUES (?, ?)', ('Asha', 95))

# Save the inserted row.
connection.commit()

# Read the row back from the table.
cursor.execute('SELECT id, name, score FROM learners')
row = cursor.fetchone()

# Print the result for confirmation.
print(row)

# Close the resources after use.
cursor.close()
connection.close()

## Concept 1: Understanding Connections

A connection represents the active link between Python and the SQLite database.

### Variations
- `sqlite3.connect(':memory:')` creates a temporary database in RAM
- `sqlite3.connect('school.db')` creates or opens a file-based database

### Common mistakes
- Forgetting to close the connection
- Assuming in-memory databases remain after the program ends

In [ ]:
import sqlite3

connection = sqlite3.connect('school.db')
cursor = connection.cursor()
cursor.execute('SELECT sqlite_version()')
print('SQLite version:', cursor.fetchone()[0])
cursor.close()
connection.close()

## Concept 2: Creating Tables and Schema Design

A table schema defines how data is organized.

### Important schema parts
- table name
- column names
- data types
- primary keys
- constraints such as `NOT NULL`, `UNIQUE`, and `DEFAULT`

### Edge cases
SQLite is flexible with types, but clear design still matters.

In [ ]:
import sqlite3

connection = sqlite3.connect('academy_sqlite.db')
cursor = connection.cursor()
cursor.execute('''
    CREATE TABLE IF NOT EXISTS students (
        student_id INTEGER PRIMARY KEY AUTOINCREMENT,
        name TEXT NOT NULL,
        email TEXT UNIQUE,
        score INTEGER DEFAULT 0
    )
''')
connection.commit()
print('students table created')
cursor.close()
connection.close()

## Concept 3: Inserting Data Safely

Insertion adds rows to a table.

### Best practice
Use placeholders to keep values separate from the SQL statement.

### Optimized variation
Use `executemany()` for batch inserts.

In [ ]:
import sqlite3

connection = sqlite3.connect('academy_sqlite.db')
cursor = connection.cursor()

cursor.execute(
    'INSERT INTO students (name, email, score) VALUES (?, ?, ?)',
    ('Ravi', 'ravi@example.com', 88)
)

rows = [
    ('Mina', 'mina@example.com', 91),
    ('Omar', 'omar@example.com', 84),
]
cursor.executemany(
    'INSERT INTO students (name, email, score) VALUES (?, ?, ?)',
    rows
)

connection.commit()
print('Batch size:', len(rows))
cursor.close()
connection.close()

## Concept 4: Reading Data with `SELECT`

Reading data is one of the most common tasks.

### Fetch methods
- `fetchone()`
- `fetchmany(size)`
- `fetchall()`

### Edge cases
- `fetchone()` returns `None` when there is no row
- `fetchall()` may use more memory for very large result sets

In [ ]:
import sqlite3

connection = sqlite3.connect('academy_sqlite.db')
cursor = connection.cursor()
cursor.execute('SELECT name, score FROM students ORDER BY score DESC')
print('First row:', cursor.fetchone())
print('Next two rows:', cursor.fetchmany(2))
print('Remaining rows:', cursor.fetchall())
cursor.close()
connection.close()

## Concept 5: Updating Rows

Updating changes existing data.

### Common mistake
Forgetting the `WHERE` clause can update every row.

In [ ]:
import sqlite3

connection = sqlite3.connect('academy_sqlite.db')
cursor = connection.cursor()
cursor.execute('UPDATE students SET score = ? WHERE name = ?', (90, 'Ravi'))
connection.commit()
print('Rows updated:', cursor.rowcount)
cursor.close()
connection.close()

## Concept 6: Deleting Rows

Deleting removes rows from a table.

### Safe habit
Use a `SELECT` first to confirm what will be deleted.

In [ ]:
import sqlite3

connection = sqlite3.connect('academy_sqlite.db')
cursor = connection.cursor()
cursor.execute('DELETE FROM students WHERE name = ?', ('Omar',))
connection.commit()
print('Rows deleted:', cursor.rowcount)
cursor.close()
connection.close()

## Concept 7: Transactions, Commit, and Rollback

Transactions group related operations.

### Why they matter
If one step fails, you may want all related steps undone.

In [ ]:
import sqlite3

connection = sqlite3.connect(':memory:')
cursor = connection.cursor()
cursor.execute('CREATE TABLE accounts (name TEXT PRIMARY KEY, balance INTEGER)')
cursor.executemany(
    'INSERT INTO accounts VALUES (?, ?)',
    [('Asha', 1000), ('Bharat', 500)]
)

try:
    cursor.execute('UPDATE accounts SET balance = balance - 200 WHERE name = ?', ('Asha',))
    cursor.execute('UPDATE accounts SET balance = balance + 200 WHERE name = ?', ('Bharat',))
    connection.commit()
except sqlite3.Error as error:
    connection.rollback()
    print('Transaction failed:', error)

cursor.execute('SELECT name, balance FROM accounts ORDER BY name')
print(cursor.fetchall())
cursor.close()
connection.close()

## Concept 8: Error Handling

Database operations can fail because of:
- duplicate values
- syntax mistakes
- invalid table names
- constraint violations

Use `try/except/finally` to keep code safe and readable.

In [ ]:
import sqlite3

try:
    connection = sqlite3.connect(':memory:')
    cursor = connection.cursor()
    cursor.execute('CREATE TABLE users (email TEXT UNIQUE)')
    cursor.execute('INSERT INTO users VALUES (?)', ('same@example.com',))
    cursor.execute('INSERT INTO users VALUES (?)', ('same@example.com',))
    connection.commit()
except sqlite3.IntegrityError as error:
    connection.rollback()
    print('Integrity error:', error)
finally:
    cursor.close()
    connection.close()

## Concept 9: Constraints and Data Validation

Constraints protect data quality.

### Common constraints
- `PRIMARY KEY`
- `UNIQUE`
- `NOT NULL`
- `CHECK`
- `DEFAULT`

### Edge case
`NULL` is not the same as `0` or an empty string.

In [ ]:
import sqlite3

connection = sqlite3.connect(':memory:')
cursor = connection.cursor()
cursor.execute('''
    CREATE TABLE employees (
        id INTEGER PRIMARY KEY,
        name TEXT NOT NULL,
        age INTEGER CHECK(age >= 18),
        department TEXT DEFAULT 'General'
    )
''')

try:
    cursor.execute('INSERT INTO employees (name, age) VALUES (?, ?)', ('Neo', 16))
    connection.commit()
except sqlite3.IntegrityError as error:
    connection.rollback()
    print('Constraint prevented insert:', error)

cursor.close()
connection.close()

## Concept 10: Using `sqlite3.Row`

By default, rows are returned as tuples.

### Better readability
Set `connection.row_factory = sqlite3.Row` to access columns by name.

In [ ]:
import sqlite3

connection = sqlite3.connect(':memory:')
connection.row_factory = sqlite3.Row
cursor = connection.cursor()
cursor.execute('CREATE TABLE products (id INTEGER PRIMARY KEY, name TEXT, price REAL)')
cursor.execute('INSERT INTO products (name, price) VALUES (?, ?)', ('Notebook', 45.5))
connection.commit()
cursor.execute('SELECT id, name, price FROM products')
row = cursor.fetchone()
print(row['name'], row['price'])
cursor.close()
connection.close()

## Concept 11: Joins

Joins combine data from multiple related tables.

### Why it matters
Real applications rarely keep everything in one table.

In [ ]:
import sqlite3

connection = sqlite3.connect(':memory:')
cursor = connection.cursor()
cursor.execute('CREATE TABLE students (student_id INTEGER PRIMARY KEY, name TEXT)')
cursor.execute('CREATE TABLE marks (student_id INTEGER, subject TEXT, score INTEGER)')
cursor.executemany('INSERT INTO students VALUES (?, ?)', [(1, 'Asha'), (2, 'Ravi')])
cursor.executemany(
    'INSERT INTO marks VALUES (?, ?, ?)',
    [(1, 'Python', 92), (1, 'SQL', 89), (2, 'Python', 85)]
)
connection.commit()
cursor.execute('''
    SELECT students.name, marks.subject, marks.score
    FROM students
    JOIN marks ON students.student_id = marks.student_id
    ORDER BY students.name
''')
print(cursor.fetchall())
cursor.close()
connection.close()

## Concept 12: Aggregation and Grouping

Aggregation creates summaries from detailed data.

### Common functions
- `COUNT()`
- `SUM()`
- `AVG()`
- `MIN()`
- `MAX()`

In [ ]:
import sqlite3

connection = sqlite3.connect(':memory:')
cursor = connection.cursor()
cursor.execute('CREATE TABLE marks (name TEXT, score INTEGER)')
cursor.executemany(
    'INSERT INTO marks VALUES (?, ?)',
    [('Asha', 90), ('Asha', 91), ('Ravi', 80)]
)
connection.commit()
cursor.execute('SELECT name, AVG(score) FROM marks GROUP BY name ORDER BY name')
print(cursor.fetchall())
cursor.close()
connection.close()

## Concept 13: Searching and Filtering

Filtering helps retrieve only the data you need.

### Useful SQL operators
- `WHERE`
- `LIKE`
- `IN`
- `BETWEEN`
- `ORDER BY`

In [ ]:
import sqlite3

connection = sqlite3.connect(':memory:')
cursor = connection.cursor()
cursor.execute('CREATE TABLE students (name TEXT, score INTEGER)')
cursor.executemany(
    'INSERT INTO students VALUES (?, ?)',
    [('Asha', 92), ('Ravi', 88), ('Mina', 95), ('Lina', 91)]
)
connection.commit()
cursor.execute('SELECT name, score FROM students WHERE score >= ? ORDER BY score DESC', (90,))
print(cursor.fetchall())
cursor.close()
connection.close()

## Concept 14: Reusable Helper Functions

Helper functions reduce duplication and make code easier to maintain.

In [ ]:
import sqlite3

def get_connection():
    connection = sqlite3.connect(':memory:')
    cursor = connection.cursor()
    cursor.execute('CREATE TABLE IF NOT EXISTS students (name TEXT, score INTEGER)')
    connection.commit()
    cursor.close()
    return connection

connection = get_connection()
print('Database ready')
connection.close()

## Concept 15: Context Manager Style

Using `with sqlite3.connect(...) as connection:` can make code cleaner.

### Benefit
The connection is committed automatically when the block succeeds.

In [ ]:
import sqlite3
from pathlib import Path

db_path = Path('context_sqlite_demo.db')
with sqlite3.connect(db_path) as connection:
    cursor = connection.cursor()
    cursor.execute('CREATE TABLE IF NOT EXISTS students (name TEXT, score INTEGER)')
    cursor.execute('DELETE FROM students')
    cursor.execute('INSERT INTO students VALUES (?, ?)', ('Lina', 94))

with sqlite3.connect(db_path) as connection:
    cursor = connection.cursor()
    cursor.execute('SELECT name, score FROM students')
    print(cursor.fetchone())

## Concept 16: Upsert Pattern

An upsert means insert a row if it does not exist, otherwise update it.

### SQLite variation
You can use `ON CONFLICT`.

In [ ]:
import sqlite3

connection = sqlite3.connect(':memory:')
cursor = connection.cursor()
cursor.execute('CREATE TABLE products (name TEXT PRIMARY KEY, price INTEGER)')
cursor.execute('INSERT INTO products VALUES (?, ?)', ('Pen', 15))
cursor.execute('''
    INSERT INTO products (name, price)
    VALUES (?, ?)
    ON CONFLICT(name) DO UPDATE SET price = excluded.price
''', ('Pen', 18))
connection.commit()
cursor.execute('SELECT name, price FROM products WHERE name = ?', ('Pen',))
print(cursor.fetchone())
cursor.close()
connection.close()

## Concept 17: Pagination with `LIMIT` and `OFFSET`

Pagination is useful when you do not want to load all rows at once.

In [ ]:
import sqlite3

page_size = 3
page_number = 2
offset = (page_number - 1) * page_size

connection = sqlite3.connect(':memory:')
cursor = connection.cursor()
cursor.execute('CREATE TABLE items (value TEXT)')
cursor.executemany(
    'INSERT INTO items VALUES (?)',
    [('row1',), ('row2',), ('row3',), ('row4',), ('row5',), ('row6',), ('row7',)]
)
connection.commit()
cursor.execute('SELECT value FROM items ORDER BY value LIMIT ? OFFSET ?', (page_size, offset))
print(cursor.fetchall())
cursor.close()
connection.close()

## Concept 18: Exporting Rows as Dictionaries

Applications often need query results in dictionary form for APIs or reports.

In [ ]:
import sqlite3

connection = sqlite3.connect(':memory:')
connection.row_factory = sqlite3.Row
cursor = connection.cursor()
cursor.execute('CREATE TABLE students (name TEXT, score INTEGER)')
cursor.executemany('INSERT INTO students VALUES (?, ?)', [('Asha', 92), ('Ravi', 88)])
connection.commit()
cursor.execute('SELECT name, score FROM students ORDER BY name')
result = [dict(row) for row in cursor.fetchall()]
print(result)
cursor.close()
connection.close()

## Concept 19: Mini Practical Example

A local results system is a good SQLite project.

### Workflow
- create table
- insert rows
- avoid duplicates
- read a report

In [ ]:
import sqlite3

connection = sqlite3.connect(':memory:')
cursor = connection.cursor()
cursor.execute('''
    CREATE TABLE students (
        student_id INTEGER PRIMARY KEY AUTOINCREMENT,
        name TEXT NOT NULL,
        email TEXT UNIQUE,
        score INTEGER NOT NULL
    )
''')
cursor.execute(
    'INSERT INTO students (name, email, score) VALUES (?, ?, ?)',
    ('Lina', 'lina@example.com', 94)
)
connection.commit()
cursor.execute('SELECT name, score FROM students ORDER BY score DESC')
print(cursor.fetchall())
cursor.close()
connection.close()

## Concept 20: Best Practices Summary

### Best practices
- use placeholders
- validate data before insertion
- commit intentionally
- roll back on failure
- close resources properly
- avoid keeping very long-lived connections in simple scripts
- design tables carefully

SQLite is a strong foundation for learning relational database work in Python.

In [ ]:
best_practices = [
    'Use placeholders for safety',
    'Commit only when needed',
    'Rollback after failures',
    'Close cursor and connection',
    'Keep SQL readable',
    'Use helper functions for repeated work',
]

for item in best_practices:
    print(item)